In [ ]:
import os 
import sys
import zarr
import numpy as np
from scipy.ndimage import convolve
from scipy import stats
from scipy.signal import fftconvolve

pythonPackagePath = os.path.abspath(r'D:\LLSM-CME-ANALYSIS\Final\src')
sys.path.append(pythonPackagePath)

In [ ]:
# Load the zarr file
base_dir = r'Z:\Abhi\LLSM_Analysis'
zarr_file_directory = 'controlOS_analysis/zarr_file/all_channels_data'
zarr_full_path = os.path.join(base_dir, zarr_file_directory)

zarr_file = zarr.open(zarr_full_path, mode='r')

In [ ]:
channel_to_detect = 2
time_point = 0
# Extract the data for the specified channel and time point
detection_data = zarr_file[time_point, channel_to_detect, :, :, :]

In [ ]:
def multiscale_adaptive_thresholding(volume, voxel_to_print, scales=[1.25, 1.75, 2.25], significance_level=0.05, use_exact_t=True):
    """
    Stage 1: Multiscale adaptive thresholding for 3D spot detection.
    
    For each scale and each voxel, fits a Gaussian with center fixed at that voxel,
    then tests if the amplitude is statistically significant.
    
    Parameters:
    -----------
    volume : ndarray (z, y, x)
        3D image data
    scales : list
        Gaussian standard deviations in voxels
    significance_level : float
        P-value threshold for significance testing
    
    Returns:
    --------
    vote_map : ndarray
        Sum of binary masks across scales (0 to len(scales))
    masks : dict
        Binary masks for each scale
    """
    
    # Initialize outputs
    vote_map = np.zeros(volume.shape, dtype=np.uint8)  # uint8 saves memory since max=3
    masks = {}
    
    # Pre-compute Gaussian kernels for all scales (major speed optimization)
    kernels = {}
    kernel_sums = {}
    kernel_sum2s = {}
    
    for σ in scales:
        # Create 3D Gaussian kernel
        # Size = ±3σ captures 99.7% of Gaussian
        kernel_radius = int(np.ceil(4 * σ))
        
        
        # Create coordinate grids
        z, y, x = np.ogrid[-kernel_radius:kernel_radius+1,
                          -kernel_radius:kernel_radius+1, 
                          -kernel_radius:kernel_radius+1]
        
        # Compute Gaussian (no normalization needed since we fit A and c)
        kernel = np.exp(-(x**2 + y**2 + z**2) / (2 * σ**2))
        
        
        # Pre-compute sums for linear system (massive speed gain)
        kernel_sums[σ] = np.sum(kernel)        # Σg
        kernel_sum2s[σ] = np.sum(kernel**2)    # Σg²
        kernels[σ] = kernel
        
        # Count non-negligible kernel elements for statistics
        kernel_n = np.sum(kernel > 0.01)  # Effective support size
        kernel_sums['n_' + str(σ)] = kernel_n
        
    # A_dict = {}
    # c_dict = {}
    # Process each scale
    for σ in scales:
        print(f"Processing scale σ={σ:.2f}")
        
        kernel = kernels[σ]
        Σg = kernel_sums[σ]
        Σg2 = kernel_sum2s[σ]
        n = kernel_sums['n_' + str(σ)]
        print(f" n: {n}")        
        # Compute convolutions (fastest way to get sums at each voxel)
        # These replace the explicit loops in the paper's equations

        # Σf = convolve(volume, np.ones_like(kernel), mode='constant')  # Σf at each voxel
        # Σgf = convolve(volume, kernel, mode='constant')  # Σ(g*f) at each voxel
        # Σf2 = convolve(volume**2, np.ones_like(kernel), mode="constant")  # needed for variance
        Σf  = fftconvolve(volume, np.ones_like(kernel), mode="same") ### Keep in mind that rounding errors can occur with this method
        Σgf = fftconvolve(volume, kernel, mode="same")
        Σf2 = fftconvolve(volume**2, np.ones_like(kernel), mode="same")  # needed for variance


        # Solve linear system at each voxel (vectorized for speed)
        # From paper: [Σg²  Σg ] [A] = [Σgf]
        #            [Σg   n  ] [c]   [Σf ]
        denominator = n * Σg2 - Σg**2  # Determinant of 2x2 system
        
        # Avoid division by zero
        if abs(denominator) < 1e-10:
            print(f"  Warning: Singular system for σ={σ}, skipping")
            masks[σ] = np.zeros(volume.shape, dtype=bool)
            continue
        
        # Solve for amplitude A at each voxel (vectorized)
        A = (n * Σgf - Σg * Σf) / denominator
        print(f"  Amplitude A at voxel {voxel_to_print}: {A[voxel_to_print]}")


        # Solve for background c at each voxel
        c = (Σf - Σg * A) / n
        print(f"  Background c at voxel {voxel_to_print}: {c[voxel_to_print]}")


        # Residual sum of squares
        RSS = Σf2 - 2 * c * Σf - 2 * A * Σgf + c**2 * n + 2 * A * c * Σg + A**2 * Σg2
        print(f"  Residual sum of squares RSS at voxel {voxel_to_print}: {RSS[voxel_to_print]}")

        # residual_variance = np.maximum(0, RSS / (n - 1)) # Claude suggests n-2, but Aguet llsmtools uses n-1
        residual_variance = np.maximum(0, RSS / (n - 2)) # Claude suggests n-2, but Aguet llsmtools uses n-1
        print(f"  Residual variance at voxel {voxel_to_print}: {residual_variance[voxel_to_print]}")

        # sigma_e2 = RSS/(n-3) # From Aguet llsmtools
        # print(f"  sigma_e2 at voxel {voxel_to_print}: {sigma_e2[voxel_to_print]}")

        # Standard error of A
        σ_A = np.sqrt(residual_variance * n / denominator) # n is element (1,1) of (JJt)-1
        # σ_A = np.sqrt(sigma_e2 * n / denominator) # From Aguet llsmtools
        print(f"  Std. error of A (σ_A) at voxel {voxel_to_print}: {σ_A[voxel_to_print]}")


        t_stat = np.divide(A, σ_A, out=np.zeros_like(A), where=σ_A > 0)
        # print t_stat at a defined voxel
        print(f"  t-stat at voxel {voxel_to_print}: {t_stat[voxel_to_print]}")

        # Critical value
        if use_exact_t:
            t_critical = stats.t.ppf(1 - significance_level/2, df=n-2)
            print(t_critical)
        else:
            t_critical = stats.norm.ppf(1 - significance_level/2)

        mask = (t_stat > t_critical) & (A > 0)
        masks[σ] = mask
        vote_map += mask.astype(np.uint8)

    return vote_map, masks

In [ ]:
# Usage example:
if __name__ == "__main__":

    
    # Convert to float32 for computation (balance memory vs precision)
    detection_data_copy = detection_data.astype(np.float32)
    
    # Run adaptive thresholding
    # scales = [1.25, 1.75, 2.25]  # MATLAB 1.25:0.5:2.25 → Python list
    scales = [1.25]
    vote_map, masks = multiscale_adaptive_thresholding(
        detection_data_copy, (0,484,278),
        scales=scales,
        significance_level=0.05,
        # n_jobs = 1  # Use all available cores (or set to specific number if needed
    )
    
    # Analyze results
    print(f"\\nVote map statistics:")
    print(f"  Voxels with 0 votes: {np.sum(vote_map == 0):,}")
    print(f"  Voxels with 1 vote:  {np.sum(vote_map == 1):,}")
    print(f"  Voxels with 2 votes: {np.sum(vote_map == 2):,}")
    print(f"  Voxels with 3 votes: {np.sum(vote_map == 3):,}")
    
    # Identify candidate regions
    candidate_mask = vote_map >= 1
    print(f"\\nTotal candidate voxels: {np.sum(candidate_mask):,}")

In [ ]:
def get_candidate_coordinates(masks):
    """
    Extract coordinates of candidate spots from binary mask.
    
    Parameters:
    -----------
    masks : ndarray
        Binary mask of candidate spots
    
    Returns:
    --------
    coordinates : list of tuples
        List of (z, y, x) coordinates of candidate spots
    """
    coordinates = np.argwhere(masks)
    return coordinates.tolist()

In [ ]:
get_candidate_coordinates(masks[1.25])

In [ ]:
### 1) Which spots are detected with fft but not convolve, for scale 1.25. 
### What are the p values for these spots, with convolve?  
### Answer: the discrepancy is due to the fit A values being 0 for the convolve method, 
### but very small (but non-zero) for the fft method. Still unclear which is the right method to use.

### 2) Is the t-statistic being calculated correctly?
### Answer: 

### 3) Are we detecting a reasonable number of spots? Is it possible that the final number of
### detected spots is greater than the number of candidate voxels in the vote map?